# Análisis exploratorio: hábitos de escucha musical por país (Last.fm, 2005–2014)

## Objetivo
Conocer y preparar los datos de escuchas de Last.fm para analizar cómo se
diferencian los hábitos de escucha musical entre países.

## Datos
- **Culture-Aware Music Recommendation Dataset** (Zangerle, Pichl y Schedl, 2020),
  subconjunto del dataset LFM-1b de Last.fm. Fuente: https://zenodo.org/records/3477842
- **Nombres de artistas** del LFM-1b. Fuente: https://zenodo.org/record/3964506

Los archivos originales (.tsv) fueron convertidos previamente a formato parquet
con el script `paso2_a_parquet.py`.

## Herramientas
- **DuckDB**: para consultar los datos con SQL. Se eligió porque la tabla de
  escuchas tiene 351 millones de filas y no entra en memoria con pandas.
- **pandas**: para manejar los resultados de las consultas (tablas chicas).
- **matplotlib**: para los gráficos.

## 1. Preparación del entorno

In [1]:
# --- Librerías ---
import duckdb                    # motor SQL que lee archivos parquet sin cargarlos enteros en memoria
import pandas as pd              # para trabajar con los resultados como DataFrames
import matplotlib.pyplot as plt  # para graficar

# --- Conexión a DuckDB ---
# connect() sin argumentos crea una base de datos en memoria (no se guarda ningún archivo).
# Solo la usamos para consultar los parquet.
con = duckdb.connect()

# Limitamos la memoria que puede usar DuckDB.
# Si una consulta necesita más, usa el disco en vez de trabar la computadora.
con.execute("SET memory_limit = '4GB'")

#El timestamp del dataset está en "unix time": segundos desde el 1/1/1970 en UTC (según el README).
# DuckDB muestra las fechas en la zona horaria de la computadora, salvo que se indique otra.
# Fijamos UTC para que las fechas no dependan de dónde se ejecute el notebook.
# y todas las horas aparecerían corridas.
con.execute("SET TimeZone = 'UTC'")

# --- Vistas: un nombre corto para cada archivo ---
# Una vista NO copia los datos: es como un "acceso directo" al archivo.
# Así en las consultas podemos escribir FROM users en vez de FROM 'data/parquet/users.parquet'.
ruta = "data/parquet/"

con.execute(f"CREATE OR REPLACE VIEW events            AS SELECT * FROM '{ruta}events.parquet'")
con.execute(f"CREATE OR REPLACE VIEW users             AS SELECT * FROM '{ruta}users.parquet'")
con.execute(f"CREATE OR REPLACE VIEW artists           AS SELECT * FROM '{ruta}artists.parquet'")
con.execute(f"CREATE OR REPLACE VIEW acoustic_features AS SELECT * FROM '{ruta}acoustic_features.parquet'")
con.execute(f"CREATE OR REPLACE VIEW hofstede          AS SELECT * FROM '{ruta}hofstede.parquet'")
con.execute(f"CREATE OR REPLACE VIEW happiness         AS SELECT * FROM '{ruta}happiness.parquet'")

# --- Función auxiliar ---
# Recibe una consulta SQL (como texto), la ejecuta y devuelve el resultado como DataFrame de pandas.
# La usamos para no repetir con.sql(...).df() en cada celda.
def q(consulta):
    return con.sql(consulta).df()

# Si se ve la tabla con "todo listo" la conexión funciona.
q("SELECT 'todo listo' AS estado")

,estado
0,todo listo


## 2. Conocer y explorar los datos

Antes de analizar, revisamos cada tabla para entender qué contiene:
- **2.1** cuántas filas tiene cada tabla,
- **2.2** qué columnas tiene y de qué tipo es cada una,
- **2.3** cómo se ven las primeras filas.

In [2]:
# Lista con los nombres de las vistas que creamos en el paso 1
tablas = ["events", "users", "artists", "acoustic_features", "hofstede", "happiness"]

# Para cada tabla contamos las filas con COUNT(*).
# fetchone()[0] devuelve directamente el número (en vez de una tabla de una sola celda).
for tabla in tablas:
    filas = con.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"{tabla:20s} {filas:>15,} filas")   # :20s y :>15, solo alinean el texto y agregan separador de miles

events                   351,469,333 filas
users                         55,190 filas
artists                    3,190,371 filas
acoustic_features          3,471,884 filas
hofstede                          47 filas
happiness                      1,562 filas


In [3]:
# DESCRIBE muestra cada columna de la tabla con su tipo de dato
# equivalente de df.info() en pandas.
# display() muestra cada resultado como tabla dentro del notebook.
for tabla in tablas:
    print(f"--- {tabla} ---")
    df_desc = con.execute(f"DESCRIBE {tabla}").df()
    display(df_desc[["column_name", "column_type"]])  # nos quedamos solo con nombre y tipo

--- events ---


,column_name,column_type
0,user_id,INTEGER
1,artist_id,INTEGER
2,album_id,INTEGER
3,track_id,INTEGER
4,timestamp,BIGINT


--- users ---


,column_name,column_type
0,user_id,BIGINT
1,country,VARCHAR
2,age,BIGINT
3,gender,VARCHAR
4,playcount,BIGINT
5,registered_unixtime,BIGINT


--- artists ---


...

,column_name,column_type
0,track_id,BIGINT
1,danceability,DOUBLE
2,energy,DOUBLE
3,key,DOUBLE
4,loudness,DOUBLE
5,mode,DOUBLE
6,speechiness,DOUBLE
7,acousticness,DOUBLE
8,instrumentalness,DOUBLE
9,liveness,DOUBLE


--- hofstede ---


,column_name,column_type
0,no,BIGINT
1,ctr,VARCHAR
2,country,VARCHAR
3,power_distance,VARCHAR
4,individualism,VARCHAR
5,masculinity,VARCHAR
6,uncertainty_avoidance,VARCHAR
7,long_term_orientation,BIGINT
8,indulgence,VARCHAR


--- happiness ---


,column_name,column_type
0,country,VARCHAR
1,year,BIGINT
2,Life Ladder,DOUBLE
3,Log GDP per capita,DOUBLE
4,Social support,DOUBLE
5,Healthy life expectancy at birth,DOUBLE
6,Freedom to make life choices,DOUBLE
7,Generosity,DOUBLE
8,Perceptions of corruption,DOUBLE
9,Positive affect,DOUBLE


In [4]:
# LIMIT 5 trae solo 5 filas
# Es el equivalente de df.head() en pandas.
for tabla in tablas:
    print(f"--- {tabla} ---")
    display(q(f"SELECT * FROM {tabla} LIMIT 5"))

--- events ---


,user_id,artist_id,album_id,track_id,timestamp
0,31435741,21,31,53,1370977938
1,31435741,21,31,53,1370977728
2,31435741,21,31,53,1370977518
3,31435741,21,31,53,1370977308
4,31435741,21,31,53,1370977098


--- users ---


,user_id,country,age,gender,playcount,registered_unixtime
0,384,UK,35,m,42139,1035849600
1,3653,UK,31,m,18504,1041033600
2,4813,US,43,m,640,1050364800
3,5069,AT,30,m,31867,1051488000
4,5213,UK,37,n,392,1052265600


--- artists ---


...

,track_id,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,16075483,0.570,0.716,3.0,-5.957,0.0,0.4490,0.184000,0.0000,0.3390,0.7990,121.719
1,19741900,0.218,0.208,9.0,-10.370,0.0,0.0336,0.965000,0.0557,0.0851,0.0429,168.227
2,299887,0.533,0.400,7.0,-12.368,1.0,0.0253,0.902000,0.9100,0.1110,0.5640,131.729
3,7891581,0.657,0.591,6.0,-8.501,0.0,0.0563,0.045100,0.5890,0.1240,0.1520,140.832
4,3869192,0.545,0.720,2.0,-8.585,1.0,0.0441,0.000798,0.0000,0.2090,0.8020,176.048


--- hofstede ---


,no,ctr,country,power_distance,individualism,masculinity,uncertainty_avoidance,long_term_orientation,indulgence
0,1,AR,Argentina,49,46,56,86,20,62
1,2,AT,Austria,11,55,79,70,60,63
2,3,AU,Australia,38,90,61,51,21,71
3,4,BE,Belgium,65,75,54,94,82,57
4,5,BG,Bulgeria,70,30,40,85,69,16


--- happiness ---


,country,year,Life Ladder,Log GDP per capita,Social support,Healthy life expectancy at birth,Freedom to make life choices,Generosity,Perceptions of corruption,Positive affect,Negative affect,Confidence in national government,Democratic Quality,Delivery Quality,Standard deviation of ladder by country-year,Standard deviation/Mean of ladder by country-year,GINI index (World Bank estimate),"GINI index (World Bank estimate), average 2000-15","gini of household income reported in Gallup, by wp5-year"
0,Afghanistan,2008,3.723590,7.168690,0.450662,49.209663,0.718114,0.181819,0.881686,0.517637,0.258195,0.612072,-1.929690,-1.655084,1.774662,0.476600,NaN,NaN,NaN
1,Afghanistan,2009,4.401778,7.333790,0.552308,49.624432,0.678896,0.203614,0.850035,0.583926,0.237092,0.611545,-2.044093,-1.635025,1.722688,0.391362,NaN,NaN,0.441906
2,Afghanistan,2010,4.758381,7.386629,0.539075,50.008961,0.600127,0.137630,0.706766,0.618265,0.275324,0.299357,-1.991810,-1.617176,1.878622,0.394803,NaN,NaN,0.327318
3,Afghanistan,2011,3.831719,7.415019,0.521104,50.367298,0.495901,0.175329,0.731109,0.611387,0.267175,0.307386,-1.919018,-1.616221,1.785360,0.465942,NaN,NaN,0.336764
4,Afghanistan,2012,3.782938,7.517126,0.520637,50.709263,0.530935,0.247159,0.775620,0.710385,0.267919,0.435440,-1.842996,-1.404078,1.798283,0.475367,NaN,NaN,0.344540


**Que es una fila y claves para unir:**

| Tabla | Una fila es... | Se une por |
|---|---|---|
| `events` | una escucha (usuario escuchó un tema en un momento) | `user_id`, `artist_id`, `track_id` |
| `users` | un usuario | `user_id` |
| `artists` | un artista | `artist_id` |
| `acoustic_features` | un tema | `track_id` |
| `hofstede`, `happiness` | un país | nombre o código de país (no es el mismo formato que `users.country`) |

In [5]:
#de todas las escuchas cuantos artistas distintos aparecen
q("SELECT approx_count_distinct(artist_id) AS artistas_escuchados_aprox FROM events")

,artistas_escuchados_aprox
0,318190


La tabla de artistas incluye los 3,19 millones de artistas del dataset original, pero solo unos 370 mil (11,6%) aparecen en las escuchas analizadas. Esto se debe a que el dataset conserva solo las canciones que pudieron encontrarse en Spotify. Como consecuencia, los datos representan principalmente a artistas conocidos/populares, y los artistas locales o independientes quedan subrepresentados.

In [6]:
# Qué valores toma cada columna de texto y cuántas veces aparece cada uno
# (equivalente a value_counts() de pandas)

# Género
display(q("SELECT gender, COUNT(*) AS usuarios FROM users GROUP BY gender ORDER BY usuarios DESC"))

# Países: cuántos hay en total y los 20 con más usuarios
display(q("SELECT COUNT(DISTINCT country) AS cantidad_paises FROM users"))
display(q("SELECT country, COUNT(*) AS usuarios FROM users GROUP BY country ORDER BY usuarios DESC LIMIT 20"))

,gender,usuarios
0,m,36506
1,f,13937
2,n,4663
3,,84


,cantidad_paises
0,208


,country,usuarios
0,US,10255
1,RU,5024
2,DE,4578
3,UK,4534
4,PL,4408
5,BR,3886
6,FI,1409
7,NL,1375
8,ES,1243
9,SE,1231


In [7]:
# Primera y última escucha, convertidas de unix time a fecha legible
q("""
    SELECT to_timestamp(MIN("timestamp")) AS primera_escucha,
           to_timestamp(MAX("timestamp")) AS ultima_escucha
    FROM events
""")

,primera_escucha,ultima_escucha
0,1970-01-01 00:00:36+00:00,2014-09-01 23:56:08+00:00


In [8]:
# Cantidad de escuchas por año.
# year() saca el año de la fecha; así vemos si hay años "imposibles" (antes de 2005)
# y cuántas escuchas tienen.
por_anio = q("""
    SELECT year(to_timestamp("timestamp")) AS anio,
           COUNT(*) AS escuchas
    FROM events
    GROUP BY anio
    ORDER BY anio
""")
por_anio

,anio,escuchas
0,1970,1
1,2005,786963
2,2006,2870944
3,2007,7111575
4,2008,12657501
5,2009,21707665
6,2010,34585255
7,2011,55158881
8,2012,90705031
9,2013,91813023


### Resumen exploracion de datos

**Tablas, granularidad y tamaño**

| Tabla | Cada fila representa... | Filas |
|---|---|---|
| events | una escucha: un usuario reprodujo una canción en un momento | 351.469.333 |
| users | un usuario de Last.fm | 55.190 |
| artists | un artista del dataset LFM-1b completo (no solo los escuchados) | 3.190.371 |
| acoustic_features | una canción con sus descriptores de audio | [2.1, aprox. 3,47 millones] |
| hofstede | un país con sus dimensiones culturales | [2.1, aprox. 47] |
| happiness | un país en un año de encuesta | [2.1] |

**Relaciones entre tablas**
- `events` se une con `users` por `user_id`, con `artists` por `artist_id` y con `acoustic_features` por `track_id`.
- `users` se une con `hofstede` por el código de país de 2 letras (`country` = `ctr`).
- `happiness` identifica los países por su nombre en inglés, por lo que necesita una tabla de equivalencias para unirse con `users`.

**Tipos de atributos**

| Atributo | Tipo |
|---|---|
| timestamp, registered_unixtime | Temporal |
| country, gender, artist_name | Categórico (nominal) |
| age | Cuantitativo (discreto) |
| danceability, energy, valence, acousticness, instrumentalness, speechiness, liveness, tempo, loudness | Cuantitativo (continuo) |
| key, mode | Categórico (key es cíclico: después de 11 vuelve a 0) |
| Life Ladder, dimensiones de Hofstede | Cuantitativo |
| user_id, artist_id, album_id, track_id | Identificadores (claves para unir tablas) |

**Observaciones**
- **Cobertura:** de los 3,19 millones de artistas de la tabla `artists`, aproximadamente 370.549 aparecen en las escuchas.
- **Países:** los usuarios pertenecen a 208 países, con una distribución muy concentrada: Estados Unidos (10.255), Rusia (5.024) y Alemania (4.578) son los que más usuarios tienen.Argentina tiene 282 usuarios.
- **Género:** además de "m" y "f", aparece el valor "n" (4663 veces), no documentado en el README.
- **Período:** el período declarado es enero 2005 – agosto 2014, pero la primera escucha registrada tiene fecha 01/01/1970 y la última 01/09/2014. Es una timestamp inválido.
- **events:** un mismo usuario puede escuchar la misma canción en loop. Las filas no están ordenadas por fecha.
- **users:** hay usuarios registrados en 2002–2003, antes del período de escuchas.
- **acoustic_features:** `key` y `mode` están guardados como decimales, pero son variables categóricas.
- **hofstede:** el código de país coincide con el de `users`. Se detectó un error de tipeo ("Bulgeria")
- **happiness:** no todos los países tienen datos desde el mismo año (Afganistán, desde 2008). Hay valores faltantes en las columnas de GINI.

## 3. Calidad de los datos

- **3.1** valores nulos,
- **3.2** valores faltantes "escondidos" (vacíos o no documentados),
- **3.3** valores fuera de rango,
- **3.4** duplicados,
- **3.5** claves que no cruzan entre tablas.

In [9]:
#3.1 valores nulos
# COUNT(*) cuenta todas las filas y COUNT(columna) solo las que NO son nulas,
# así que la diferencia es la cantidad de nulos de esa columna.
def contar_nulos(tabla):
    columnas = q(f"DESCRIBE {tabla}")["column_name"]                        # lista de columnas de la tabla
    partes = [f'COUNT(*) - COUNT("{col}") AS "{col}"' for col in columnas]  # un conteo por columna
    resultado = q(f"SELECT {', '.join(partes)} FROM {tabla}").T             # .T: una fila por columna
    resultado.columns = ["nulos"]
    return resultado

for tabla in ["users", "acoustic_features", "hofstede", "happiness"]:
    print(f"--- {tabla} ---")
    display(contar_nulos(tabla))

--- users ---


,nulos
user_id,0
country,0
age,0
gender,0
playcount,0
registered_unixtime,0


--- acoustic_features ---


,nulos
track_id,0
danceability,2655
energy,373
key,294
loudness,0
mode,294
speechiness,2654
acousticness,401
instrumentalness,454
liveness,759


--- hofstede ---


,nulos
no,0
ctr,0
country,0
power_distance,0
individualism,0
masculinity,0
uncertainty_avoidance,0
long_term_orientation,0
indulgence,0


--- happiness ---


,nulos
country,0
year,0
Life Ladder,0
Log GDP per capita,27
Social support,13
Healthy life expectancy at birth,9
Freedom to make life choices,29
Generosity,80
Perceptions of corruption,90
Positive affect,18


In [10]:
contar_nulos("events") #los eventos son millones

,nulos
user_id,0
artist_id,0
album_id,0
track_id,0
timestamp,0


acoustic_features:  pocos nulos.

-El peor caso es valence, con 3.595 nulos sobre unos 3,47 millones de canciones: el 0,1%.
-En el README aclara que los features de una canción pueden estar incompletos porque la API de Spotify no los devolvió.
-tempo y loudness tienen 0 nulos, pero pueden tener faltantes escondidos ejemplo que sea 0

happiness: muchos nulos, pero no se van a usar esas columnas

In [11]:
# 3.2 Faltantes escondidos en users
# - País vacío: un texto '' no cuenta como nulo, pero tampoco es un país.
# - Género 'n': el README dice que solo puede ser 'm' o 'f', pero hay una n
# - Género con otro valor
q("""
    SELECT
        COUNT(*) FILTER (WHERE TRIM(country) = '')            AS pais_vacio,
        COUNT(*) FILTER (WHERE gender = 'n')                  AS genero_n,
        COUNT(*) FILTER (WHERE gender NOT IN ('m', 'f', 'n')) AS genero_otro_valor,
        COUNT(*)                                              AS total_usuarios
    FROM users
""")

,pais_vacio,genero_n,genero_otro_valor,total_usuarios
0,0,4663,84,55190


In [12]:
# Todos los valores de género y cuántos usuarios tiene cada uno
q("SELECT gender, COUNT(*) AS usuarios FROM users GROUP BY gender ORDER BY usuarios DESC")

,gender,usuarios
0,m,36506
1,f,13937
2,n,4663
3,,84


In [13]:
#3.3.a valores fuera de rango

# se consideran válidas las edades entre 10 y 90 años.
# Esta consulta muestra qué outliers aparecen y cuántos usuarios tiene cada uno.
q("""
    SELECT age, COUNT(*) AS usuarios
    FROM users
    WHERE age < 10 OR age > 90
    GROUP BY age
    ORDER BY usuarios DESC
""")

,age,usuarios
0,-1,12139
1,0,29
2,111,29
3,2,26
4,3,25
5,112,24
6,1,21
7,6,17
8,4,16
9,101,14


In [14]:
#3.3.b audio features
# Un tempo de 0 BPM o un volumen de -60 dB (silencio total) no son posibles en una canción real:
# son valores que Spotify no pudo calcular y quedaron como 0 o como el mínimo de la escala.
q("""
    SELECT
        COUNT(*) FILTER (WHERE tempo = 0)      AS tempo_cero,
        COUNT(*) FILTER (WHERE loudness = -60) AS loudness_silencio,
        COUNT(*)                               AS total_canciones
    FROM acoustic_features
""")

,tempo_cero,loudness_silencio,total_canciones
0,2651,209,3471884


In [15]:
#3.3.c Fechas

# tabla por anio calculada en la exploracion de datos
# Nos quedamos con los años fuera del período 2005-2014 y calculamos qué porcentaje del total son.
fuera = por_anio[(por_anio["anio"] < 2005) | (por_anio["anio"] > 2014)]

print("Escuchas fuera de 2005-2014:", fuera["escuchas"].sum())
print("Porcentaje del total:", round(fuera["escuchas"].sum() / por_anio["escuchas"].sum() * 100, 4), "%")
fuera

Escuchas fuera de 2005-2014: 1
Porcentaje del total: 0.0 %


,anio,escuchas
0,1970,1


In [16]:
#3.3.d Valores atípicos: usuarios con actividad extrema

# Escuchas por usuario y por día.
# date_trunc('day', ...) recorta la fecha al día (sin hora), para contar cuántas escuchas hubo cada día.
# Después nos quedamos con el máximo de cada usuario: su "día más activo".
por_usuario_dia = q("""
    SELECT user_id, MAX(escuchas_dia) AS max_escuchas_en_un_dia
    FROM (
        SELECT user_id,
               date_trunc('day', to_timestamp("timestamp")) AS dia,
               COUNT(*) AS escuchas_dia
        FROM events
        GROUP BY user_id, dia
    )
    GROUP BY user_id
""")

# describe() muestra mínimo, máximo, promedio y percentiles: así vemos si hay valores extremos
por_usuario_dia["max_escuchas_en_un_dia"].describe(percentiles=[0.5, 0.9, 0.99])

count    55143.000000
mean       102.233230
std         93.336707
min          1.000000
50%         85.000000
90%        185.000000
99%        399.000000
max       3534.000000
Name: max_escuchas_en_un_dia, dtype: float64

In [17]:
# Cuántos usuarios superan las 500 escuchas en un mismo día (unas 33 horas de música a 4 minutos por tema)
(por_usuario_dia["max_escuchas_en_un_dia"] > 500).sum()

np.int64(307)

- La mayoría de los usuarios tiene un máximo diario razonable (mediana: 85 escuchas en su día más activo;
  99% por debajo de 399). Sin embargo, 307 usuarios superan las 500 escuchas en un mismo día,
  con un máximo de 3.534. Estos valores no corresponden podrian ser bots o de importaciones masivas de escuchas.
- 47 usuarios no tienen ninguna escucha en el dataset (55.143 usuarios con escuchas sobre 55.190).

aclaracion de que es lo que Last.fm considera 1 escucha: si dura mas de 30 segundos o si se escucho al menos la mitad. fuente: https://www.last.fm/api/scrobbling

In [18]:
# Umbral basado en la regla de Last.fm: cada escucha requiere al menos la mitad de la canción.
# Con canciones de ~4 minutos, son al menos 2 minutos por escucha: máximo 720 escuchas en 24 horas.
(por_usuario_dia["max_escuchas_en_un_dia"] > 720).sum()

np.int64(140)

In [19]:
# Días en los que un usuario superó las 720 escuchas, y cuántas escuchas suman en total
q("""
    SELECT COUNT(*)          AS dias_anomalos,
           SUM(escuchas_dia) AS escuchas_en_dias_anomalos
    FROM (
        SELECT user_id,
               date_trunc('day', to_timestamp("timestamp")) AS dia,
               COUNT(*) AS escuchas_dia
        FROM events
        GROUP BY user_id, dia
    )
    WHERE escuchas_dia > 720
""")

,dias_anomalos,escuchas_en_dias_anomalos
0,664,727379.0


- Esos 140 usuarios acumulan 664 días anómalos, que suman 727.379 escuchas (0,21% del total),
  con un promedio de 1.095 escuchas por día. Se eliminarán las escuchas de esos días, conservando
  el resto de la actividad de los usuarios: la pérdida de datos es mínima y evita que estos registros
  distorsionen el análisis temporal.

In [20]:
#3.3.e consistencia de cuantas escuchas hay antes de haberse registrado a Last.fm
#puede ser que la fecha este mal

# Cantidad de escuchas con fecha anterior a la fecha de registro del usuario
q("""
    SELECT COUNT(*) AS escuchas_antes_del_registro,
           COUNT(DISTINCT e.user_id) AS usuarios_afectados
    FROM events e
    JOIN users u ON e.user_id = u.user_id
    WHERE e."timestamp" < u.registered_unixtime
""")

,escuchas_antes_del_registro,usuarios_afectados
0,2963,760


In [21]:
# 3.4 Duplicados
# COUNT(*) cuenta todas las filas; COUNT(DISTINCT ...) cuenta los IDs sin repetir.
# Si los dos números dan igual, no hay IDs duplicados.
q("""
    SELECT
        (SELECT COUNT(*) FROM users)                             AS filas_users,
        (SELECT COUNT(DISTINCT user_id) FROM users)              AS usuarios_distintos,
        (SELECT COUNT(*) FROM acoustic_features)                 AS filas_canciones,
        (SELECT COUNT(DISTINCT track_id) FROM acoustic_features) AS canciones_distintas,
        (SELECT COUNT(*) FROM artists)                           AS filas_artistas,
        (SELECT COUNT(DISTINCT artist_id) FROM artists)          AS artistas_distintos
""")

,filas_users,usuarios_distintos,filas_canciones,canciones_distintas,filas_artistas,artistas_distintos
0,55190,55190,3471884,3471884,3190371,3190371


In [22]:
# Escuchas duplicadas de verdad: mismo usuario, misma canción y mismo timestamp exacto.
# Agrupamos por esas tres columnas y contamos los grupos que aparecen más de una vez.
q("""
    SELECT COUNT(*) AS grupos_duplicados, SUM(veces - 1) AS filas_sobrantes
    FROM (
        SELECT user_id, track_id, "timestamp", COUNT(*) AS veces
        FROM events
        GROUP BY user_id, track_id, "timestamp"
        HAVING COUNT(*) > 1
    )
""")

,grupos_duplicados,filas_sobrantes
0,0,NaN


No hay valores duplicados

In [23]:
#3.5.a escuchas sin usuario, artista o cancion

# LEFT JOIN conserva TODAS las escuchas, aunque no hagan match en la otra tabla.
# Si no la encuentran, las columnas de la otra tabla quedan en NULL.
# Contamos cuántas escuchas quedaron sin match en cada tabla, (ideal 0).
q("""
    SELECT
        COUNT(*) FILTER (WHERE u.user_id   IS NULL) AS escuchas_sin_usuario,
        COUNT(*) FILTER (WHERE a.artist_id IS NULL) AS escuchas_sin_artista,
        COUNT(*) FILTER (WHERE f.track_id  IS NULL) AS escuchas_sin_audio_features
    FROM events e
    LEFT JOIN users u             ON e.user_id   = u.user_id
    LEFT JOIN artists a           ON e.artist_id = a.artist_id
    LEFT JOIN acoustic_features f ON e.track_id  = f.track_id
""")

,escuchas_sin_usuario,escuchas_sin_artista,escuchas_sin_audio_features
0,0,0,0


In [24]:
#3.5.b paises de users en hoefstede

# Ambas tablas usan el código de país de 2 letras (users.country y hofstede.ctr).
# Primero vemos cómo figura el Reino Unido en hofstede: en users aparece como 'UK'.
display(q("SELECT ctr, country FROM hofstede WHERE ctr IN ('UK', 'GB')"))

# Después contamos cuántos usuarios y países quedan sin datos de Hofstede.
q("""
    SELECT COUNT(*)                  AS usuarios_sin_hofstede,
           COUNT(DISTINCT u.country) AS paises_sin_hofstede
    FROM users u
    LEFT JOIN hofstede h ON u.country = h.ctr
    WHERE h.ctr IS NULL
""")

,ctr,country
0,UK,Great Britain


,usuarios_sin_hofstede,paises_sin_hofstede
0,1881,161


- `hofstede` utiliza el mismo código de país que `users` (incluido "UK" para el Reino Unido),
  por lo que ambas tablas se unen directamente.
- Hofstede incluye solo 47 de los 208 países presentes en `users`. Sin embargo, esos 47 países
  concentran mas del 95% de los usuarios: los 161 países sin datos culturales suman solo 1.881 usuarios.
- Para el mapa será necesario convertir "UK" al código estándar ISO "GB".